# Testing data Augmentation

In [ ]:
import sys
from pathlib import Path
# fixing imports for provided_sources/ modules
sys.path.append(str(Path().resolve().parents[1]))

import copy
import pprint
import wandb
import gc

import torch
import torch.nn as nn
import src.cnn.cnn_paths as paths
import src.cnn.cnn_models as cnn_models
from src.cnn.configs import ExperimentConfig
from src.cnn.cnn_registry import build_model
from src.cnn.cnn_utils import *

In [ ]:
augmentation_variants = {
    "no_aug": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ]),

    # only apply a random crop and a horizontal flip
    "light_aug": transforms.Compose([
        transforms.Resize((240, 240)),
        transforms.RandomCrop((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
    ]),
    
    # apply crop, rotation, horizontal flip and color jitter
    "medium_aug": transforms.Compose([
        transforms.Resize((240, 240)),
        transforms.RandomCrop((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(8),
        transforms.ColorJitter(
            brightness=0.1,
            contrast=0.1,
            saturation=0.1,
            hue=0.02,
        ),
        transforms.ToTensor(),
    ]),

    # apply zoom, horizontal flip, shear and color jitter
    "strong_aug": transforms.Compose([
        transforms.RandomResizedCrop(
            size=224,
            scale=(0.7, 1.0),
            ratio=(0.85, 1.15),
        ),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomAffine(
            degrees=12,
            translate=(0.10, 0.10),
            scale=(0.90, 1.10),
            shear=8,
        ),
        transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.03,
        ),
        transforms.ToTensor(),
    ]),

    # apply strong Vertical flip, allow bad rotation angles, very strong color jitter --> results are images that don't represent expected images of animals.
    "bad_aug": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomVerticalFlip(p=0.9),
        transforms.RandomRotation(180),
        transforms.ColorJitter(
            brightness=0.8,
            contrast=0.8,
            saturation=0.8,
            hue=0.5,
        ),
        transforms.RandomResizedCrop(
            224,
            scale=(0.05, 1.0),
            ratio=(0.2, 3.0),
        ),
        transforms.ToTensor(),
    ]),
}

all_results = {}

for aug_name, train_transform in augmentation_variants.items():
    print(f"\n==== Training with augmentation: {aug_name} ====\n")

    # see configs.py
    cfg = ExperimentConfig()

    # =========================================================
    # DATASET CONFIG
    # =========================================================
    cfg.dataset.dataset_dir = paths.DATASET_DIR # see cnn_paths.py
    cfg.dataset.train_subdir = "train"
    cfg.dataset.val_subdir = "validate"
    cfg.dataset.test_subdir = None   # or "test" if you have it
    cfg.dataset.image_size = 224

    # leave transforms as None -> default Resize + ToTensor pipeline
    cfg.dataset.train_transform = train_transform
    cfg.dataset.eval_transform = transforms.Compose([ #resestting to default transforms for evaluation
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

    # optional normalization
    cfg.dataset.normalize_mean = None
    cfg.dataset.normalize_std = None


    # =========================================================
    # MODEL CONFIG
    # IMPORTANT: name is saved in MODEL_REGISTRY -> cnn_registry.py
    # use @register_model("depth_cnn") decorator to add new models to the registry and make them available by name in the config
    # =========================================================
    cfg.model.name = "depth_cnn_fc" # use model from fc testing with only one fully connected layer in the header
    cfg.model.kwargs = {
        "depth": 12,
        "in_channels": 3,
        "num_classes": 10,
        "base_channels": 32,
        "max_channels": 256,
        "dropout_conv": 0.0,
        "dropout_fc": 0.5,
        "inputsize": 224,
        "fc_layers": 1,
        # construction decreasing header to keep numbers of parameters in a reasonable range
        "hidden1": 128,
        "hidden2": 64,
        "hidden3": 32,
        "hidden4": 16,
    }


    # =========================================================
    # DATALOADER CONFIG
    # =========================================================
    cfg.loader.batch_size = 32 # reduced batchsize from learnings in previous experiments
    cfg.loader.num_workers = 0
    cfg.loader.pin_memory = True
    cfg.loader.train_shuffle = True
    cfg.loader.eval_shuffle = False
    cfg.loader.drop_last_train = False
    cfg.loader.drop_last_eval = False


    # =========================================================
    # LOSS CONFIG
    # =========================================================
    cfg.loss.cls = nn.CrossEntropyLoss
    cfg.loss.kwargs = {}


    # =========================================================
    # OPTIMIZER CONFIG
    # =========================================================
    cfg.optimizer.cls = torch.optim.SGD
    cfg.optimizer.kwargs = {
        "lr": 0.01, # from previous testings
        "momentum": 0.9, # default value for momentum
        "weight_decay": 1e-4, # deafault value
    }


    # =========================================================
    # SCHEDULER CONFIG
    # Example: Reduce LR when validation loss plateaus
    # =========================================================
    cfg.scheduler.cls = None
    cfg.scheduler.kwargs = {
        # "mode": "min",
        # "factor": 0.5,
        # "patience": 2,
    }
    cfg.scheduler.step_metric = None

    # =========================================================
    # EARLY STOPPING CONFIG
    # OPTIONAL
    # For noisy validation curves use:
    # early_stopping_patience = 7
    # early_stopping_min_delta = 0.001
    # =========================================================

    cfg.train.early_stopping = False # use early stopping based on validation accuracy to prevent overfitting and save training time
    cfg.train.early_stopping_patience = 5
    cfg.train.early_stopping_min_delta = 0.0


    # =========================================================
    # TRAIN CONFIG
    # =========================================================
    cfg.train.epochs = 100
    cfg.train.device = str(get_device("auto"))
    cfg.train.non_blocking = True
    cfg.train.use_amp = (cfg.train.device == "cuda") # suggestion from ChatGPT for large batch sizes to save memory and speed up training
    cfg.train.grad_clip_norm = None
    cfg.train.best_metric = "val/accuracy"
    cfg.train.best_mode = "max"
    cfg.train.seed = 42


    # =========================================================
    # W&B CONFIG
    # =========================================================
    cfg.wandb.enabled = True
    cfg.wandb.project = "MPW-CNN"
    cfg.wandb.entity = "MSE_DeLearn_SPR26"
    cfg.wandb.mode = "online"   # "online", "offline", or "disabled" for no logging

    # NOTE: use meaningful names for runs, so the difference is clear
    cfg.wandb.run_name = f"{aug_name}_ModelBatchSize{cfg.loader.batch_size}_depth{cfg.model.kwargs['depth']}_momentum{cfg.optimizer.kwargs['momentum']}_lr{cfg.optimizer.kwargs['lr']}_fc_layers{cfg.model.kwargs['fc_layers']}_e{cfg.train.epochs}"

    # NOTE: use meaningful grouping, for example, by model or by task.
    # E.g. task "Data augmentation" => experiments on light-mid-heavy augmentation
    cfg.wandb.group = "Data_augmentation"
    cfg.wandb.job_type = "train"

    # NOTE: use meaningful tags to filter runs in UI
    cfg.wandb.tags = ["Data_augmentation"]
    cfg.wandb.notes = "Full config live run"

    cfg.wandb.log_epoch_metrics = True
    cfg.wandb.log_every_n_epochs = 1

    cfg.wandb.metric_allowlist = {
        "train/loss",
        "train/accuracy",
        "val/loss",
        "val/accuracy",
        "gap/accuracy",
        "gap/loss",
        "lr",
    }

    cfg.wandb.summary_allowlist = {
        "best_epoch",
        "best_metric_name",
        "best_metric_value",
        "train_final/loss",
        "train_final/accuracy",
        "val_final/loss",
        "val_final/accuracy",
    }

    cfg.wandb.watch_model = False
    cfg.wandb.watch_log = "all"
    cfg.wandb.watch_log_freq = 100

    # =========================================================
    # CONFUSION MATRIX CONFIG
    # =========================================================

    cfg.wandb.log_confusion_matrix = True
    cfg.wandb.confusion_matrix_split = "val"
    cfg.wandb.confusion_matrix_title = f"{aug_name} - Confusion Matrix"
    cfg.wandb.confusion_matrix_split_table = False

    # =========================================================
    # RUN EXECUTION
    # =========================================================
    try:
        # see cnn_utils.py
        datasets_dict = load_datasets(cfg.dataset)

        train_dataset = datasets_dict["train"]
        val_dataset = datasets_dict["val"]
        test_dataset = datasets_dict.get("test")

        train_loader = make_train_loader(train_dataset, cfg)
        val_loader = make_eval_loader(val_dataset, cfg)

        test_loader = None
        if test_dataset is not None:
            test_loader = make_eval_loader(test_dataset, cfg)

        model = build_model(cfg.model)

        print(model)
        print(f"Trainable parameters: {get_num_parameters(model):,}")

        model, history, result = train_and_evaluate_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            test_loader=test_loader,
            cfg=cfg,
            run_name=cfg.wandb.run_name,
        )

        pprint.pprint(result)


        all_results[aug_name] = copy.deepcopy(result)

        cfg_eval = copy.deepcopy(cfg)
        cfg_eval.train.device = "cpu"

        model_cpu = model.to("cpu").eval()

        if torch.backends.mps.is_available():
            torch.mps.synchronize()
            torch.mps.empty_cache()

        loader_for_cm = test_loader if test_loader is not None else val_loader
        split_name = "test" if test_loader is not None else "val"
        class_names = loader_for_cm.dataset.classes

        y_true, y_pred = predict_loader(model_cpu, loader_for_cm, cfg_eval)

        print("structure of predictions/true labels:")
        print("true label counts:", Counter(y_true))
        print("pred label counts:", Counter(y_pred))
        print("class_names:", class_names)

        print("Unique y_true:", sorted(set(y_true)))
        print("Unique y_pred:", sorted(set(y_pred)))


        fig_counts = make_confusion_matrix_fig(
            y_true=y_true,
            y_pred=y_pred,
            class_names=class_names,
            normalize=None,
            title=f"{split_name.capitalize()} Confusion Matrix (counts)",
        )

        fig_norm = make_confusion_matrix_fig(
            y_true=y_true,
            y_pred=y_pred,
            class_names=class_names,
            normalize="true",
            title=f"{split_name.capitalize()} Confusion Matrix (normalized)",
        )

        with wandb.init(
            project=cfg.wandb.project,
            entity=cfg.wandb.entity,
            mode=cfg.wandb.mode,
            id=result["wandb_run_id"], # add to the same run!
            resume="must",              # for that, run is resumed
            job_type="log_confusion_matrix",
        ):
            try:
                log_confusion_matrix(
                    y_true=y_true,
                    y_pred=y_pred,
                    class_names=class_names,
                    key=f"{split_name}/confusion_matrix_chart",
                    title=f"{split_name.capitalize()} Confusion Matrix",
                    split_table=False,
                )
            except Exception as e:
                wandb.log({
                    f"{split_name}/confusion_matrix_chart_error": str(e)
                })
                print("W&B custom confusion matrix failed:", e)

            wandb.log({
                f"{split_name}/confusion_matrix_counts": wandb.Image(fig_counts),
                f"{split_name}/confusion_matrix_normalized": wandb.Image(fig_norm),
            })

            wandb.finish()

        # free memory before next run
        del model, history, result
        gc.collect() # 
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"OOM during run '{aug_name}', skipping...")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        else:
            raise

# =========================================================
# FINAL RESULTS
# =========================================================

print("\n" + "=" * 100)
print("SUMMARY (DATA AUGMENTATION COMPARISON)")
print("=" * 100)

for aug_name, res in all_results.items():
    print(
        f"{aug_name:>12} | "
        f"best_val_acc={res['best_metric_value']:.4f} | "
        f"best_epoch={res['best_epoch']} | "
        f"val_final_acc={res.get('val_final/accuracy', float('nan')):.4f}"
    )